# Notebook 2 — Polarization, EIS, and CV Analysis

This notebook consumes the processed experiment created by **Notebook 1 v1.0** and generates publication-style figures and analysis tables for the working electrode measured versus Hg/HgO.

It includes:

- OCP, activation CP, and preconditioning CP;
- multistep chronopotentiometry staircase;
- raw and iR-corrected polarization curves;
- Hg/HgO and RHE potential scales;
- Tafel analysis;
- Nyquist evolution for all spectra and for spectra from 10 mA cm⁻² onward;
- initial/final EIS replicate comparisons;
- CV evolution using the last complete cycle from every CV;
- initial-versus-final CV comparison;
- journal-ready PNG, SVG, and PDF exports.

No full-cell quantities are calculated.


In [ ]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.ticker import AutoMinorLocator

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 1. Configuration

In [ ]:
# -------- USER CONFIGURATION --------
PROCESSED_DIR = Path(
    r"C:\Users\edr2299\OneDrive - The University of Texas at Austin\Documents\Research\Electrochemistry Analysis\Results\Processed_data_Notebook_1"
)

ANALYSIS_DIR = PROCESSED_DIR / "analysis_outputs"
FIGURES_DIR = ANALYSIS_DIR / "figures"
TABLES_DIR = ANALYSIS_DIR / "tables"

# Reference-electrode conversion:
# E_RHE = E_Hg/HgO + HG_HGO_TO_RHE_OFFSET_V
#
# Concentrated KOH is non-ideal. Replace this value with the offset established
# by your laboratory calibration or the convention chosen for the manuscript.
HG_HGO_TO_RHE_OFFSET_V = 0.926

# OER equilibrium potential on the RHE scale.
OER_EQUILIBRIUM_V_RHE = 1.229

# CP averaging window: final fraction of each step.
CP_STEADY_STATE_FRACTION = 0.20

# Tafel fitting limits in mA cm^-2. Modify after visually inspecting the data.
TAFEL_J_MIN_MA_CM2 = 10.0
TAFEL_J_MAX_MA_CM2 = 100.0

# Nyquist detail figure starts at this current density.
NYQUIST_DETAIL_MIN_J_MA_CM2 = 10.0

# Figure export settings.
FIGURE_DPI = 600
SAVE_PNG = True
SAVE_SVG = True
SAVE_PDF = True

for folder in [ANALYSIS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Processed directory:", PROCESSED_DIR)
print("Exists:", PROCESSED_DIR.exists())
print("Analysis directory:", ANALYSIS_DIR)


## 2. Publication-style plotting helpers

In [ ]:
def set_publication_style():
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.fontsize": 7.5,
        "axes.linewidth": 1.0,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.minor.width": 0.7,
        "ytick.minor.width": 0.7,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.minor.size": 2.5,
        "ytick.minor.size": 2.5,
        "legend.frameon": False,
        "figure.dpi": 120,
        "savefig.bbox": "tight",
    })

def format_axes(ax, xlabel=None, ylabel=None, title=None):
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
    return ax

def add_panel_label(ax, label):
    ax.text(-0.16, 1.06, label, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top", ha="left")

def save_figure(fig, stem):
    paths = []
    if SAVE_PNG:
        p = FIGURES_DIR / f"{stem}.png"
        fig.savefig(p, dpi=FIGURE_DPI)
        paths.append(p)
    if SAVE_SVG:
        p = FIGURES_DIR / f"{stem}.svg"
        fig.savefig(p)
        paths.append(p)
    if SAVE_PDF:
        p = FIGURES_DIR / f"{stem}.pdf"
        fig.savefig(p)
        paths.append(p)
    return paths

def metadata_box(ax, lines, loc=(0.98, 0.04)):
    text = "\n".join(str(x) for x in lines if x)
    ax.text(
        loc[0], loc[1], text,
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=6.8,
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white",
              "edgecolor": "0.55", "linewidth": 0.6, "alpha": 0.88},
    )

set_publication_style()


## 3. Load Notebook 1 outputs

In [ ]:
manifest_path = PROCESSED_DIR / "experiment_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Could not find {manifest_path}. Run Notebook 1 first and confirm PROCESSED_DIR."
    )

with manifest_path.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

metadata = manifest["experiment_metadata"]
files_manifest = manifest["files"]
steps_manifest = manifest["polarization_steps"]

files_by_name = {item["filename"]: item for item in files_manifest}
files_by_role = {}
for item in files_manifest:
    files_by_role.setdefault(item["role"], []).append(item)

area_cm2 = float(metadata["geometric_area_cm2"])

print("Experiment:", metadata.get("experiment_name"))
print("Working electrode:", metadata.get("working_electrode"))
print("Electrolyte:", metadata.get("electrolyte"))
print("Geometric area:", area_cm2, "cm²")
print("Files in manifest:", len(files_manifest))
print("Polarization steps:", len(steps_manifest))


In [ ]:
def read_relative_csv(relative_path):
    if not relative_path:
        return None
    path = PROCESSED_DIR / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def first_table(item):
    csvs = item.get("table_csv_files", [])
    if not csvs:
        raise ValueError(f"No table CSV registered for {item['filename']}")
    return read_relative_csv(csvs[0])

def selected_cv(item):
    return read_relative_csv(item.get("selected_cv_csv"))

def numeric_column(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    raise KeyError(f"None of {candidates} found. Available columns: {list(df.columns)}")

def role_one(role):
    items = files_by_role.get(role, [])
    if len(items) != 1:
        raise ValueError(f"Expected one {role} file; found {len(items)}")
    return items[0]

def clean_numeric(df, columns):
    out = df.copy()
    for col in columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out.dropna(subset=columns)


## 4. OCP, activation CP, and preconditioning CP

In [ ]:
def cp_trace(item):
    df = first_table(item)
    tcol = numeric_column(df, ["T", "Time"])
    vcol = numeric_column(df, ["Vf", "V", "Ewe"])
    icol = numeric_column(df, ["Im", "I", "Idc"])
    out = clean_numeric(df, [tcol, vcol, icol])
    return pd.DataFrame({
        "time_s": out[tcol],
        "potential_V_HgHgO": out[vcol],
        "current_A": out[icol],
    })

ocp_item = role_one("ocp")
activation_item = role_one("activation_cp")
pre_item = role_one("preconditioning_cp")

ocp = first_table(ocp_item)
ocp_t = numeric_column(ocp, ["T", "Time"])
ocp_v = numeric_column(ocp, ["Vf", "V", "Ewe"])
ocp = clean_numeric(ocp, [ocp_t, ocp_v])

activation = cp_trace(activation_item)
pre_cp = cp_trace(pre_item)

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.35))
axes[0].plot(ocp[ocp_t] / 60, ocp[ocp_v], lw=1.2)
format_axes(axes[0], "Time / min", r"$E_{WE}$ / V vs Hg/HgO", "Open-circuit potential")
add_panel_label(axes[0], "a")

axes[1].plot(activation["time_s"] / 3600, activation["potential_V_HgHgO"], lw=1.0)
format_axes(axes[1], "Time / h", r"$E_{WE}$ / V vs Hg/HgO", "Activation chronopotentiometry")
add_panel_label(axes[1], "b")
metadata_box(axes[1], [
    f"j = {activation['current_A'].median()*1000/area_cm2:.1f} mA cm$^{{-2}}$",
    f"t = {activation['time_s'].max()/3600:.1f} h",
])

axes[2].plot(pre_cp["time_s"] / 60, pre_cp["potential_V_HgHgO"], lw=1.0)
format_axes(axes[2], "Time / min", r"$E_{WE}$ / V vs Hg/HgO", "Preconditioning chronopotentiometry")
add_panel_label(axes[2], "c")
metadata_box(axes[2], [
    f"j = {pre_cp['current_A'].median()*1000/area_cm2:.2f} mA cm$^{{-2}}$",
    f"t = {pre_cp['time_s'].max()/60:.1f} min",
])

fig.tight_layout()
save_figure(fig, "01_ocp_activation_preconditioning")
plt.show()


## 5. Reconstruct the multistep CP staircase and steady-state polarization table

In [ ]:
step_rows = []
staircase_parts = []
elapsed_s = 0.0

for step in steps_manifest:
    cp_item = files_by_name[step["cp_file"]]
    trace = cp_trace(cp_item).sort_values("time_s").reset_index(drop=True)
    duration_s = float(trace["time_s"].max() - trace["time_s"].min())
    start_idx = int(np.floor((1 - CP_STEADY_STATE_FRACTION) * len(trace)))
    steady = trace.iloc[start_idx:].copy()

    current_A = float(steady["current_A"].median())
    j_mA_cm2 = current_A * 1000 / area_cm2
    e_mean = float(steady["potential_V_HgHgO"].mean())
    e_sd = float(steady["potential_V_HgHgO"].std(ddof=1))

    local = trace.copy()
    local["global_time_s"] = elapsed_s + (local["time_s"] - local["time_s"].min())
    local["step"] = int(step["step"])
    local["j_mA_cm2"] = j_mA_cm2
    staircase_parts.append(local)
    elapsed_s += duration_s

    step_rows.append({
        "step": int(step["step"]),
        "cp_file": step["cp_file"],
        "eis_file": step["eis_file"],
        "cv_file": step["cv_file"],
        "current_A": current_A,
        "current_density_mA_cm2": j_mA_cm2,
        "potential_mean_V_HgHgO": e_mean,
        "potential_sd_V": e_sd,
        "cp_duration_s": duration_s,
        "steady_state_points": len(steady),
    })

polarization = pd.DataFrame(step_rows).sort_values("current_density_mA_cm2").reset_index(drop=True)
staircase = pd.concat(staircase_parts, ignore_index=True)

polarization.to_csv(TABLES_DIR / "polarization_raw_steady_state.csv", index=False)
display(polarization)


In [ ]:
fig, ax1 = plt.subplots(figsize=(5.8, 3.4))
ax1.plot(staircase["global_time_s"] / 60, staircase["potential_V_HgHgO"], lw=0.9)
format_axes(ax1, "Cumulative time / min", r"$E_{WE}$ / V vs Hg/HgO",
            "Multistep chronopotentiometry")

ax2 = ax1.twinx()
step_centers = []
for _, group in staircase.groupby("step", sort=True):
    x0 = group["global_time_s"].min() / 60
    x1 = group["global_time_s"].max() / 60
    j = group["j_mA_cm2"].iloc[0]
    ax2.hlines(j, x0, x1, lw=1.4)
    step_centers.append(((x0 + x1) / 2, j))
ax2.set_ylabel(r"$j$ / mA cm$^{-2}$")
ax2.tick_params(direction="in")
for spine in ax2.spines.values():
    spine.set_linewidth(1.0)

metadata_box(ax1, [
    f"{len(polarization)} current steps",
    f"{polarization['current_density_mA_cm2'].min():.2f}–{polarization['current_density_mA_cm2'].max():.1f} mA cm$^{{-2}}$",
    f"steady state = final {CP_STEADY_STATE_FRACTION*100:.0f}%",
])
fig.tight_layout()
save_figure(fig, "02_multistep_cp_staircase")
plt.show()


## 6. EIS processing and local iR correction

In [ ]:
def eis_trace(item):
    df = first_table(item)
    fcol = numeric_column(df, ["Freq", "Frequency"])
    zrcol = numeric_column(df, ["Zreal", "Zre"])
    zicol = numeric_column(df, ["Zimag", "Zim"])
    out = clean_numeric(df, [fcol, zrcol, zicol])
    result = pd.DataFrame({
        "frequency_Hz": out[fcol],
        "Zreal_ohm": out[zrcol],
        "Zimag_ohm": out[zicol],
    })
    if "Idc" in df.columns:
        result["Idc_A"] = pd.to_numeric(df.loc[out.index, "Idc"], errors="coerce").to_numpy()
    if "Vdc" in df.columns:
        result["Vdc_V"] = pd.to_numeric(df.loc[out.index, "Vdc"], errors="coerce").to_numpy()
    return result.sort_values("frequency_Hz", ascending=False).reset_index(drop=True)

def estimate_hfr(eis):
    # Use the high-frequency real-axis intercept estimated from the two points
    # bracketing -Zimag = 0. If no crossing exists, use the real component of
    # the highest-frequency point and flag the method.
    e = eis.sort_values("frequency_Hz", ascending=False).reset_index(drop=True)
    x = e["Zreal_ohm"].to_numpy(float)
    y = (-e["Zimag_ohm"]).to_numpy(float)
    for i in range(len(e) - 1):
        if y[i] == 0:
            return float(x[i]), "exact crossing"
        if y[i] * y[i + 1] < 0:
            frac = -y[i] / (y[i + 1] - y[i])
            return float(x[i] + frac * (x[i + 1] - x[i])), "linear intercept"
    return float(x[0]), "highest-frequency Zreal"

hfr_rows = []
eis_by_step = {}

for _, row in polarization.iterrows():
    item = files_by_name[row["eis_file"]]
    eis = eis_trace(item)
    hfr, method = estimate_hfr(eis)
    eis_by_step[int(row["step"])] = eis
    hfr_rows.append({
        "step": int(row["step"]),
        "eis_file": row["eis_file"],
        "current_density_mA_cm2": row["current_density_mA_cm2"],
        "HFR_ohm": hfr,
        "HFR_method": method,
    })

hfr_table = pd.DataFrame(hfr_rows)
polarization = polarization.merge(hfr_table, on=["step", "eis_file", "current_density_mA_cm2"], how="left")
polarization["iR_drop_V"] = polarization["current_A"] * polarization["HFR_ohm"]
polarization["potential_iR_corrected_V_HgHgO"] = (
    polarization["potential_mean_V_HgHgO"] - polarization["iR_drop_V"]
)
polarization["potential_raw_V_RHE"] = (
    polarization["potential_mean_V_HgHgO"] + HG_HGO_TO_RHE_OFFSET_V
)
polarization["potential_iR_corrected_V_RHE"] = (
    polarization["potential_iR_corrected_V_HgHgO"] + HG_HGO_TO_RHE_OFFSET_V
)
polarization["overpotential_iR_corrected_V"] = (
    polarization["potential_iR_corrected_V_RHE"] - OER_EQUILIBRIUM_V_RHE
)

polarization.to_csv(TABLES_DIR / "polarization_complete.csv", index=False)
hfr_table.to_csv(TABLES_DIR / "eis_hfr_by_step.csv", index=False)
display(polarization)


## 7. Polarization curves on Hg/HgO and RHE scales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.45), sharey=True)

axes[0].errorbar(
    polarization["potential_mean_V_HgHgO"],
    polarization["current_density_mA_cm2"],
    xerr=polarization["potential_sd_V"],
    marker="o", ms=3.2, lw=1.0, capsize=2,
)
format_axes(axes[0], r"$E_{WE}$ / V vs Hg/HgO", r"$j$ / mA cm$^{-2}$", "Raw polarization")
add_panel_label(axes[0], "a")

axes[1].plot(
    polarization["potential_iR_corrected_V_HgHgO"],
    polarization["current_density_mA_cm2"],
    marker="o", ms=3.2, lw=1.0,
)
format_axes(axes[1], r"$E_{WE,iR}$ / V vs Hg/HgO", None, "iR-corrected")
add_panel_label(axes[1], "b")

axes[2].plot(
    polarization["potential_iR_corrected_V_RHE"],
    polarization["current_density_mA_cm2"],
    marker="o", ms=3.2, lw=1.0,
)
format_axes(axes[2], r"$E_{WE,iR}$ / V vs RHE", None, "iR-corrected")
add_panel_label(axes[2], "c")
metadata_box(axes[2], [f"Hg/HgO → RHE offset = {HG_HGO_TO_RHE_OFFSET_V:.3f} V"])

fig.tight_layout()
save_figure(fig, "03_polarization_curves")
plt.show()


## 8. Tafel analysis

In [ ]:
tafel = polarization.loc[
    (polarization["current_density_mA_cm2"] >= TAFEL_J_MIN_MA_CM2) &
    (polarization["current_density_mA_cm2"] <= TAFEL_J_MAX_MA_CM2) &
    (polarization["current_density_mA_cm2"] > 0)
].copy()

if len(tafel) < 3:
    warnings.warn("Fewer than three points are inside the configured Tafel window.")

tafel["log10_j"] = np.log10(tafel["current_density_mA_cm2"])
x = tafel["log10_j"].to_numpy()
y = tafel["overpotential_iR_corrected_V"].to_numpy()
slope_V_dec, intercept_V = np.polyfit(x, y, 1)
fit = slope_V_dec * x + intercept_V
ss_res = np.sum((y - fit) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

tafel_summary = pd.DataFrame([{
    "j_min_mA_cm2": TAFEL_J_MIN_MA_CM2,
    "j_max_mA_cm2": TAFEL_J_MAX_MA_CM2,
    "n_points": len(tafel),
    "tafel_slope_mV_dec": slope_V_dec * 1000,
    "intercept_V": intercept_V,
    "R_squared": r_squared,
}])
tafel.to_csv(TABLES_DIR / "tafel_points.csv", index=False)
tafel_summary.to_csv(TABLES_DIR / "tafel_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(3.5, 3.0))
ax.scatter(x, y * 1000, s=22, zorder=3, label="Experimental")
xfit = np.linspace(x.min(), x.max(), 100)
ax.plot(xfit, (slope_V_dec * xfit + intercept_V) * 1000, lw=1.2, label="Linear fit")
format_axes(ax, r"$\log_{10}(j / \mathrm{mA\,cm^{-2}})$", r"$\eta_{iR}$ / mV", "Tafel analysis")
metadata_box(ax, [
    f"b = {slope_V_dec*1000:.1f} mV dec$^{{-1}}$",
    f"$R^2$ = {r_squared:.4f}",
    f"{TAFEL_J_MIN_MA_CM2:g} ≤ j ≤ {TAFEL_J_MAX_MA_CM2:g} mA cm$^{{-2}}$",
])
ax.legend(loc="best")
fig.tight_layout()
save_figure(fig, "04_tafel_analysis")
plt.show()

display(tafel_summary)


## 9. Nyquist evolution

In [ ]:
norm = colors.Normalize(
    vmin=polarization["current_density_mA_cm2"].min(),
    vmax=polarization["current_density_mA_cm2"].max(),
)
cmap = cm.viridis

def plot_nyquist_evolution(data_rows, stem, title):
    fig, ax = plt.subplots(figsize=(4.2, 3.5))
    for _, row in data_rows.iterrows():
        eis = eis_by_step[int(row["step"])]
        ax.plot(
            eis["Zreal_ohm"],
            -eis["Zimag_ohm"],
            lw=0.9,
            color=cmap(norm(row["current_density_mA_cm2"])),
        )
    format_axes(ax, r"$Z'$ / Ω", r"$-Z''$ / Ω", title)
    ax.set_aspect("equal", adjustable="datalim")
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = fig.colorbar(sm, ax=ax, pad=0.03)
    cbar.set_label(r"$j$ / mA cm$^{-2}$")
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()

plot_nyquist_evolution(
    polarization,
    "05_nyquist_evolution_all",
    "Nyquist evolution — all current steps",
)

detail = polarization[
    polarization["current_density_mA_cm2"] >= NYQUIST_DETAIL_MIN_J_MA_CM2
].copy()
plot_nyquist_evolution(
    detail,
    "06_nyquist_evolution_from_10mAcm2",
    rf"Nyquist evolution — $j \geq$ {NYQUIST_DETAIL_MIN_J_MA_CM2:g} mA cm$^{{-2}}$",
)


## 10. Initial and final EIS replicate comparison

In [ ]:
initial_eis_summary = pd.read_csv(PROCESSED_DIR / "initial_eis_mean_sd.csv")
final_eis_summary = pd.read_csv(PROCESSED_DIR / "final_eis_mean_sd.csv")

fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))

for ax, summary, title, panel in [
    (axes[0], initial_eis_summary, "Initial EIS replicates", "a"),
    (axes[1], final_eis_summary, "Final EIS replicates", "b"),
]:
    fcol = numeric_column(summary, ["frequency_Hz", "Freq"])
    zrmean = numeric_column(summary, ["Zreal_mean_ohm", "Zreal_mean"])
    zimean = numeric_column(summary, ["Zimag_mean_ohm", "Zimag_mean"])
    zrstd = next((c for c in summary.columns if c.lower() in {"zreal_sd_ohm", "zreal_std_ohm", "zreal_sd"}), None)
    zistd = next((c for c in summary.columns if c.lower() in {"zimag_sd_ohm", "zimag_std_ohm", "zimag_sd"}), None)

    ax.plot(summary[zrmean], -summary[zimean], marker="o", ms=2.2, lw=0.9)
    if zrstd and zistd:
        stride = max(1, len(summary) // 12)
        ax.errorbar(
            summary[zrmean].iloc[::stride],
            -summary[zimean].iloc[::stride],
            xerr=summary[zrstd].iloc[::stride],
            yerr=summary[zistd].iloc[::stride],
            fmt="none", capsize=1.5, lw=0.6,
        )
    format_axes(ax, r"$Z'$ / Ω", r"$-Z''$ / Ω", title)
    ax.set_aspect("equal", adjustable="datalim")
    add_panel_label(ax, panel)

fig.tight_layout()
save_figure(fig, "07_initial_final_eis_replicates")
plt.show()


## 11. CV evolution — all last complete cycles

In [ ]:
cv_sequence = []

initial_cv_item = role_one("initial_cv")
initial_cv_df = selected_cv(initial_cv_item)
cv_sequence.append(("Initial", 0.0, initial_cv_item, initial_cv_df))

for _, row in polarization.sort_values("current_density_mA_cm2").iterrows():
    item = files_by_name[row["cv_file"]]
    cv_sequence.append((
        f"{row['current_density_mA_cm2']:.2f} mA cm$^{{-2}}$",
        float(row["current_density_mA_cm2"]),
        item,
        selected_cv(item),
    ))

final_cv_item = role_one("final_cv")
final_cv_df = selected_cv(final_cv_item)
cv_sequence.append(("Final", float(polarization["current_density_mA_cm2"].max()), final_cv_item, final_cv_df))

def standardized_cv(df):
    vcol = numeric_column(df, ["Vf", "V", "Ewe"])
    icol = numeric_column(df, ["Im", "I"])
    out = clean_numeric(df, [vcol, icol])
    return pd.DataFrame({
        "potential_V_HgHgO": out[vcol],
        "potential_V_RHE": out[vcol] + HG_HGO_TO_RHE_OFFSET_V,
        "current_density_mA_cm2": out[icol] * 1000 / area_cm2,
    })

cmap_cv = cm.plasma
norm_cv = colors.Normalize(vmin=0, vmax=len(cv_sequence)-1)

fig, ax = plt.subplots(figsize=(4.7, 3.7))
for idx, (label, preceding_j, item, df) in enumerate(cv_sequence):
    cv = standardized_cv(df)
    ax.plot(
        cv["potential_V_RHE"],
        cv["current_density_mA_cm2"],
        lw=0.85,
        color=cmap_cv(norm_cv(idx)),
    )

format_axes(ax, r"$E_{WE}$ / V vs RHE", r"$j$ / mA cm$^{-2}$", "CV evolution")
sm = cm.ScalarMappable(norm=norm_cv, cmap=cmap_cv)
cbar = fig.colorbar(sm, ax=ax, pad=0.03)
cbar.set_label("Measurement order")
cbar.set_ticks([0, len(cv_sequence)-1])
cbar.set_ticklabels(["Initial", "Final"])
metadata_box(ax, [
    f"{len(cv_sequence)} CV measurements",
    "last complete cycle",
    f"Hg/HgO → RHE offset = {HG_HGO_TO_RHE_OFFSET_V:.3f} V",
])
fig.tight_layout()
save_figure(fig, "08_cv_evolution_all_steps_RHE")
plt.show()


## 12. Initial-versus-final CV

In [ ]:
initial_cv = standardized_cv(initial_cv_df)
final_cv = standardized_cv(final_cv_df)

fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
for ax, xcol, xlabel, panel in [
    (axes[0], "potential_V_HgHgO", r"$E_{WE}$ / V vs Hg/HgO", "a"),
    (axes[1], "potential_V_RHE", r"$E_{WE}$ / V vs RHE", "b"),
]:
    ax.plot(initial_cv[xcol], initial_cv["current_density_mA_cm2"], lw=1.1, label="Initial")
    ax.plot(final_cv[xcol], final_cv["current_density_mA_cm2"], lw=1.1, label="Final")
    format_axes(ax, xlabel, r"$j$ / mA cm$^{-2}$", "Initial vs final CV")
    add_panel_label(ax, panel)
    ax.legend(loc="best")

fig.tight_layout()
save_figure(fig, "09_initial_vs_final_cv")
plt.show()


## 13. Export CV evolution data and analysis summary

In [ ]:
cv_export_rows = []
for order, (label, preceding_j, item, df) in enumerate(cv_sequence):
    cv = standardized_cv(df)
    temp = cv.copy()
    temp.insert(0, "measurement_order", order)
    temp.insert(1, "measurement_label", label)
    temp.insert(2, "source_file", item["filename"])
    temp.insert(3, "preceding_current_density_mA_cm2", preceding_j)
    cv_export_rows.append(temp)

cv_evolution_table = pd.concat(cv_export_rows, ignore_index=True)
cv_evolution_table.to_csv(TABLES_DIR / "cv_evolution_all_last_complete_cycles.csv", index=False)

analysis_summary = {
    "experiment_name": metadata.get("experiment_name"),
    "working_electrode": metadata.get("working_electrode"),
    "electrolyte": metadata.get("electrolyte"),
    "geometric_area_cm2": area_cm2,
    "hg_hgo_to_rhe_offset_V": HG_HGO_TO_RHE_OFFSET_V,
    "cp_steady_state_fraction": CP_STEADY_STATE_FRACTION,
    "polarization_step_count": len(polarization),
    "current_density_min_mA_cm2": float(polarization["current_density_mA_cm2"].min()),
    "current_density_max_mA_cm2": float(polarization["current_density_mA_cm2"].max()),
    "tafel_slope_mV_dec": float(slope_V_dec * 1000),
    "tafel_R_squared": float(r_squared),
    "tafel_j_min_mA_cm2": TAFEL_J_MIN_MA_CM2,
    "tafel_j_max_mA_cm2": TAFEL_J_MAX_MA_CM2,
    "cv_measurement_count": len(cv_sequence),
}

with (ANALYSIS_DIR / "analysis_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(analysis_summary, handle, indent=2)

pd.DataFrame([analysis_summary]).to_csv(TABLES_DIR / "analysis_summary.csv", index=False)

print("Analysis complete.")
print("Figures:", FIGURES_DIR)
print("Tables:", TABLES_DIR)
display(pd.DataFrame([analysis_summary]))


## 14. Quality-control checklist

Before using the figures in a manuscript:

1. Confirm the Hg/HgO-to-RHE offset used for your exact reference electrode, temperature, and electrolyte convention.
2. Inspect the HFR intercept selected for every spectrum.
3. Confirm the steady-state CP averaging window.
4. Select a physically justified linear Tafel region rather than accepting the default limits automatically.
5. Confirm that the selected CV cycle is complete and that scan rates are being compared appropriately.
6. Treat all potentials as working-electrode potentials; do not compare them numerically with the full-cell voltages reported in the reference paper.
